# 🛡️ GTM Engineer Job Market Analysis (March 2026)

This notebook provides a deep dive into the GTM (Go-To-Market) Engineering job market. We analyze 310 high-quality technical roles extracted from Builtin.com across major tech hubs.

---

## ⚙️ Section 0: Setup and Data Loading
Loading libraries, setting the premium dark theme, and importing the structured dataset.

In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from IPython.display import display, Markdown

# Premium Aesthetics
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

# Setup chart directory
CHART_DIR = "_internal/charts"
if not os.path.exists(CHART_DIR):
    os.makedirs(CHART_DIR)

# Load Structured Data
DATA_PATH = "_internal/structured_jobs.json"
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
    df = pd.DataFrame(raw_data)

# Load Summary Stats if available
try:
    with open("_internal/summary_stats.json", 'r') as f:
        summary = json.load(f)
except FileNotFoundError:
    summary = {}

display(Markdown(f"### ✅ Data successfully loaded: {len(df)} total records analyzed."))

## 📊 Section 1: Dataset Overview
We start with a high-level view of our collection. We filtered 829 raw scraped postings down to the 310 most relevant technical GTM roles.

In [2]:
# Metrics
total_scraped = 829
df_gtm = df[df["is_gtm_technical"] == True].copy()
gtm_tech_total = len(df_gtm)
relevance_rate = (gtm_tech_total / total_scraped) * 100

display(Markdown(f"""
**Core Metrics:**
- **Total Scraped:** {total_scraped}
- **GTM Technical Roles (300 Spartans):** {gtm_tech_total}
- **Relevance Concentration:** {relevance_rate:.1f}%
"""))

# Top Companies Visualization
top_companies = df_gtm['company'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_companies.values, y=top_companies.index, palette="viridis")
plt.title("Top 10 Companies Hiring GTM Engineers", fontsize=16, weight='bold', pad=20)
plt.xlabel("Open Technical Roles", labelpad=10)
plt.ylabel("Company", labelpad=10)
plt.savefig(os.path.join(CHART_DIR, "01_top_companies.png"), bbox_inches='tight', dpi=300)
plt.show()

print(f"\nTop 5 Companies:\n{top_companies.head(5)}")

## 🧪 Section 2: Job Type Distribution
GTM Engineering is a broad field. Here we look at how many roles categorized clearly as Technical GTM versus broader ops or sales support roles.

In [3]:
dist = df['is_gtm_technical'].value_counts()
labels = ['GTM Technical (Relevant)', 'Non-Technical/Others']
colors = ['#00d4ff', '#333333']

plt.figure(figsize=(10, 6))
plt.pie(dist, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, explode=[0.1, 0], 
        textprops={'fontsize': 12, 'weight': 'bold'})
plt.title("GTM Technical Role Distribution", fontsize=16, pad=20, weight='bold')
plt.savefig(os.path.join(CHART_DIR, "02_job_type_distribution.png"), bbox_inches='tight', dpi=300)
plt.show()

## 💰 Section 3: Compensation Analysis
Analying base salary ranges across different seniority levels. This helps benchmark expectations for GTM Engineers in 2026.

In [4]:
df_gtm["salary_min"] = pd.to_numeric(df_gtm["salary_min"], errors='coerce')
df_gtm["salary_max"] = pd.to_numeric(df_gtm["salary_max"], errors='coerce')
sal_df = df_gtm.dropna(subset=["salary_min", "salary_max", "seniority"])
sal_df = sal_df[sal_df["salary_min"] > 1000] # Ignore low-value placeholders
sal_df["avg_salary"] = (sal_df["salary_min"] + sal_df["salary_max"]) / 2

seniority_order = ["Junior", "Mid", "Senior", "Lead", "Staff", "Manager", "Head"]
current_order = [o for o in seniority_order if o in sal_df["seniority"].unique()]

plt.figure(figsize=(14, 8))
sns.boxplot(data=sal_df, x="seniority", y="avg_salary", order=current_order, hue="seniority", palette="magma", legend=False)
plt.title("GTM Technical Salary Benchmarks (Annual Base USD)", fontsize=16, fontweight='bold', pad=25)
plt.ylabel("Base Salary (USD)")
plt.xlabel("Seniority Level")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.savefig(os.path.join(CHART_DIR, "03_compensation_analysis.png"), bbox_inches='tight', dpi=300)
plt.show()

# Median Salary Table
median_salaries = sal_df.groupby('seniority')['avg_salary'].median().reindex(current_order)
print("\nMedian Salary by Seniority:")
print(median_salaries.apply(lambda x: f"${x:,.0f}"))

## 🛠️ Section 4: Tech Stack (Preview)
Which tools are most frequently requested in GTM Engineering roles?

In [5]:
all_tools = []
for tools in df_gtm["tech_stack"].dropna():
    if isinstance(tools, list): all_tools.extend(tools)

tool_counts = pd.Series(all_tools).value_counts().head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x=tool_counts.values, y=tool_counts.index, hue=tool_counts.index, palette="viridis", legend=False)
plt.title("Top 15 GTM Tech Stack Tools (Frequency)", fontsize=16, fontweight='bold', pad=25)
plt.xlabel("Count (Number of Postings)")
plt.show()